# Predicting Mouse Experimental Group from Protein Expression

A machine learning project on the UCI Mice Protein Expression dataset. The goal is to predict which of the eight experimental groups a sample belongs to using protein expression levels, then identify which proteins drive the prediction.

In [1]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report

## Step 1 - Gather and look at the data

Load the dataset from the UCI Machine Learning Repository. Before changing anything, look at its shape, the column names, and the class balance. Inspecting first is what catches surprises early.

In [2]:
# Load the Mice Protein Expression dataset (id 342)
data = fetch_ucirepo(id=342)

X = data.data.features
y = data.data.targets
mouse_id = data.data.ids["MouseID"].str.split("_").str[0] # "309_1" -> "309"

print("Features shape:", X.shape)
print("Target column:", list(y.columns))
print("Numeric protein columns:", X.select_dtypes(include="number").shape[1])
print("Text columns:", list(X.select_dtypes(exclude="number").columns))
print()
print("Class counts:")
print(y["class"].value_counts())

Features shape: (1080, 80)
Target column: ['class']
Numeric protein columns: 77
Text columns: ['Genotype', 'Treatment', 'Behavior']

Class counts:
class
c-CS-m    150
c-SC-m    150
c-CS-s    135
c-SC-s    135
t-CS-m    135
t-SC-m    135
t-SC-s    135
t-CS-s    105
Name: count, dtype: int64


## Step 2 - Clean the data

Drop the three text columns, since Genotype, Treatment, and Behavior combined are exactly what defines the class label, so keeping them would leak the answer. Then drop proteins missing more than fifteen percent of their values, because filling a mostly empty column means inventing too much data.

In [3]:
# Keep only the numeric protein columns (drops Genotype, Treatment, Behavior)
X = X.select_dtypes(include="number")

# Drop proteins missing more than 15% of their values
cutoff = 0.15 * len(X)
too_many_missing = X.columns[X.isna().sum() > cutoff]
X = X.drop(columns=too_many_missing)

print("Dropped proteins:", list(too_many_missing))
print("Shape after cleaning:", X.shape)
print("Missing cells remaining:", int(X.isna().sum().sum()))

Dropped proteins: ['BAD_N', 'BCL2_N', 'H3AcK18_N', 'EGR1_N', 'H3MeK4_N']
Shape after cleaning: (1080, 72)
Missing cells remaining: 238


## Step 3 - Split into train and test, grouped by mouse

Hold back 20 percent of the data as a sealed test set that the model never sees during training. The split is done by mouse, not by row, because the dataset has fifteen repeated measurements per mouse. A random split would let the same mouse appear in both train and test, which inflates accuracy. Grouping by mouse prevents that leakage.

In [4]:
labels = y["class"]

# GroupShuffleSplit keeps every measurement of one mouse entirely on one side
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, labels, groups=mouse_id))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = labels.iloc[train_idx], labels.iloc[test_idx]

# Confirm no mouse is in both sets (should be 0)
overlap = set(mouse_id.iloc[train_idx]) & set(mouse_id.iloc[test_idx])

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("Mice in BOTH train and test (should be 0):", len(overlap))

Training samples: 855
Test samples: 225
Mice in BOTH train and test (should be 0): 0


## Step 4 - Fill the remaining missing values

Fill the scattered empty cells with each protein's median. The median is learned from the training set only, then applied to both train and test. This is the core discipline against leakage. Anything learned from data is learned from training alone, so the sealed test set never influences it.

In [5]:
imputer = SimpleImputer(strategy="median")

# fit_transform on TRAINING: learn the medians and fill training blanks
X_train = imputer.fit_transform(X_train)

# transform on TEST: fill using the training medians, no peeking
X_test = imputer.transform(X_test)

print("Training missing after fill:", int(np.isnan(X_train).sum()))
print("Test missing after fill:", int(np.isnan(X_test).sum()))
print("Training shape:", X_train.shape)

Training missing after fill: 0
Test missing after fill: 0
Training shape: (855, 72)


## Step 5 - Scale the proteins

Put every protein on the same numeric footing, with an average of zero and a spread of one. Some proteins have large values and some small, and scaling stops the model from treating a protein as important just because its raw numbers are bigger. The scaling is learned from training only, then applied to both sets.

In [6]:
scaler = StandardScaler()

# fit_transform on TRAINING: learn each protein's average and spread, then scale
X_train = scaler.fit_transform(X_train)

# transform on TEST: scale using the training values, no peeking
X_test = scaler.transform(X_test)

print("Mean of first protein after scaling (about 0):", round(X_train[:,0].mean(), 4))
print("Spread of first protein after scaling (about 1):", round(X_train[:,0].std(), 4))

Mean of first protein after scaling (about 0): 0.0
Spread of first protein after scaling (about 1): 1.0


## Step 6 - Select the most informative proteins

Keep the thirty proteins whose values differ most across the eight groups, measured with an ANOVA F test, and drop the rest as noise. Fewer, sharper features make a cleaner model and a clearer importance ranking later. The selection is learned from training only.

In [7]:
selector = SelectKBest(score_func=f_classif, k=30)

# fit_transform on TRAINING: score proteins and keep the top 30
X_train = selector.fit_transform(X_train, y_train)

# trainform on TEST: keep the same 30 proteins chosen from training
X_test = selector.transform(X_test)

print("Training shape after selection:", X_train.shape)
print("Test shape after selecton:", X_test.shape)

Training shape after selection: (855, 30)
Test shape after selecton: (225, 30)


## Step 7 - Train the Random Forest and evaluate

Train a Random Forest, which is hundreds of decision trees that each vote on the answer. Evaluate it in three ways. A baseline that always guesses the most common group shows the floor. Cross-validation grouped by mouse checks that the result is stable. The held-out test set gives the honest score on mice the model never saw.

In [8]:
# Baseline: alwaus guess the most common group
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
baseline = accuracy_score(y_test, dummy.predict(X_test))

# The Random Forest: 300 trees voting
rf = RandomForestClassifier(n_estimators=300, random_state=42)

# Cross-validation, grouped by mouse so no mouse spans two folds
train_mouse_id = mouse_id.iloc[train_idx]
gkf = GroupKFold(n_splits=5)
cv_scores = cross_val_score(rf, X_train, y_train, cv=gkf, groups=train_mouse_id)

# Train on all training data, then test on the sealed test set
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Baseline accuracy:", round(baseline, 3))
print("Cross-validated accuracy:", round(cv_scores.mean(), 3), "+/-", round(cv_scores.std(), 3))
print("Held-out test accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred, zero_division=0))

Baseline accuracy: 0.067
Cross-validated accuracy: 0.583 +/- 0.111
Held-out test accuracy: 0.653

              precision    recall  f1-score   support

      c-CS-m       0.73      0.53      0.62        60
      c-CS-s       0.07      0.13      0.09        15
      c-SC-m       1.00      0.71      0.83        45
      c-SC-s       0.54      1.00      0.70        15
      t-CS-m       0.31      0.80      0.44        15
      t-CS-s       1.00      0.30      0.46        30
      t-SC-m       1.00      1.00      1.00        15
      t-SC-s       1.00      1.00      1.00        30

    accuracy                           0.65       225
   macro avg       0.71      0.68      0.64       225
weighted avg       0.79      0.65      0.67       225



## Step 8 - Which proteins drove the predictions

Measure how much the model relies on each protein using permutation importance. Each protein's values are randomly shuffled, and the drop in accuracy is recorded. A big drop means the model depended on that protein. Permutation importance is used instead of the tree's built-in importance because it is less biased and more trustworthy.

In [9]:
# Recover the name of the 30 proteins the selector kept in Step 6
kept_mask = selector.get_support() # True/False for each of the 72
all_proteins = X.columns # the 72 protein names before the selection
kept_proteins = all_proteins[kept_mask] # the 30 that survived

# Permutation importance: shuffle each protein, measure the accuracy drop
result = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42)

importance = pd.Series(result.importances_mean, index=kept_proteins)
importance = importance.sort_values(ascending=False)

print("Top 15 proteins driving the prediction:")
print(importance.head(15).round(4))

Top 15 proteins driving the prediction:
APP_N          0.0500
SOD1_N         0.0473
ITSN1_N        0.0389
pS6_N          0.0262
CaNA_N         0.0251
Ubiquitin_N    0.0251
P38_N          0.0189
ARC_N          0.0169
pCAMKII_N      0.0162
pPKCAB_N       0.0140
S6_N           0.0131
pERK_N         0.0122
DYRK1A_N       0.0098
NUMB_N         0.0093
ERBB4_N        0.0082
dtype: float64
